# SAM 3 / SAM 3.1 room upgrade
Completed GPU experiments, gated object fusion and short temporal-mask motion checks. This notebook reviews cached results; see `research/SAM_UPGRADE_REPORT.md` for limitations and timing scope.

Recompute GPU stages separately with `.venv-sam31/Scripts/python.exe research/try_sam.py image`, `hybrid`, `video`, `video --prompt beanbag`. CPU fusion uses the original Anaconda runtime. The complete room identity map does not yet use SAM 3.1 tracks.

In [ ]:
from pathlib import Path
import sys,json
ROOT=Path.cwd()
assert (ROOT/'research').exists()
sys.path.insert(0,str(ROOT/'.packages'))
import numpy as np,pandas as pd
from IPython.display import display,Image,HTML
summary=json.loads((ROOT/'outputs/sam_trial/sam_summary.json').read_text())
display(summary)
assert summary['fusion']['accepted_refinements']+summary['fusion']['fallbacks']==summary['fusion']['proposals']

In [ ]:
display(Image(filename=str(ROOT/'outputs/sam_trial/sam_comparison.png')))
print('Compare masks and failures; detection count is not accuracy.')

In [ ]:
for prompt,filename in [('furniture','sam31_tracks.json'),('beanbag','sam31_beanbag_tracks.json')]:
    tracks=pd.DataFrame(json.loads((ROOT/'outputs/sam_trial'/filename).read_text()))
    assert set(tracks.frame)==set(range(400,490))
    assert (tracks.mask_pixels>0).all()
    display(tracks.groupby('object_id').agg(frames=('frame','nunique'),minimum_mask_pixels=('mask_pixels','min')))
display(Image(filename=str(ROOT/'outputs/sam_trial/sam31_beanbag_000450.jpg')))
motion=pd.read_csv(ROOT/'outputs/sam_trial/sam_motion_comparison.csv')
display(motion[['class_name','region','valid_pixels','score','fraction_flagged','ego_refinement_supported']])

In [ ]:
audit=pd.read_csv(ROOT/'outputs/sam_room/sam_fusion_audit.csv')
assert len(audit)==243 and audit.accepted.sum()==186
surfaces=np.load(ROOT/'outputs/sam_room/object_surfaces.npz')
assert np.isfinite(surfaces['points']).all()
objects=json.loads((ROOT/'outputs/sam_room/room_objects.json').read_text())
assert len(objects)==9
display(audit.groupby(['class_name','accepted']).size())
display(Image(filename=str(ROOT/'outputs/sam_room/room_floorplan.png')))

In [ ]:
display(Image(filename=str(ROOT/'outputs/sam_room/room_3d_cutaway.png')))
display(HTML('<a href="outputs/sam_room/room_3d_interactive.html" target="_blank">Open SAM-refined interactive 3D room</a>'))
print('Measured mesh reused. Object surfaces refined; geometry holes and scale uncertainty remain.')